# Computer Vision

Computer Vision is a huge sub domain. Many research tasks involve getting information out of a video or image. Exactly what information and how you extract and represent it can vary. Images are usually stored in some kind of numerical format where a number represents a pixel location and value (color, brightness, etc). There are a variety of mathamtical tricks you can do to extract edges, colors, shapes, patterns. Let's explore a few approaches and workflows that do this under the hood.

## Object Detection 
Object Detection simply draws a rectangular bounding box around each item and gives it a label with a confidence score. You have likely helped trained these when you solve a Captcha challenge on a website. Traditionally, the model can only classifier objects that it has a known label for.

## Semantic Segmentation
Groups pixels purely by their class label (e.g., "person," "car," "road"). It does not differentiate between individual objects of the same class. If three people are standing next to each other, semantic segmentation colours all three as one blob of "person."

## Instance Segmentation
Focuses strictly on countable objects (known as "things"). It detects each individual object and assigns it a unique identity. If three people are in a room, it outlines Person 1, Person 2, and Person 3 separately. However, it completely ignores amorphous backgrounds like the floor, sky, or grass (known as "stuff").

## Panoptic Segmentation
Combines both semantic and instance segmentation into a single, unified output. It assigns every single pixel in the image both a class label and an instance ID (if applicable). Nothing is left unclassified.

This has been grouped into:

* "Things": Countable objects with well-defined shapes (e.g., cars, people, chairs, animals, trees). These get individual instance IDs.

* "Stuff": Uncountable, amorphous background regions or textures (e.g., sky, road, water, grass, sand). These get semantic labels but no instance IDs (since you can't count "one sky" vs. "another sky").

## Vision transformers and multimodal AI

If you ask a multimodal AI model (Gemini, etc) to "segment the trees," it needs a pre-trained "tree" class. If you ask it to "segment only the trees that look sick or dying," it usually fails because it lacks ecological reasoning.

Classification: "Is there a dog in this photo?" (Single output label, zero spatial info).Object Detection: "Where are the dogs?" (Draws boxes around Dog 1 at coordinates $[x_1, y_1]$ and Dog 2 at $[x_2, y_2]$).Semantic Segmentation: "Which pixels are dog pixels vs. background?" (Colors all dog pixels red, but merges both dogs together into one shape).Instance Segmentation: "Where are the exact pixel outlines of each separate dog?" (Colors Dog 1 red and Dog 2 blue with precise pixel-perfect boundaries—essentially Object Detection + Precise Pixel Masks).Panoptic Segmentation: "Outline every dog, person, and car individually, and paint all the grass, sky, and road background pixels too." (Total scene understanding).

Gemini excels here because it combines visual spatial perception with semantic world knowledge:

Zero-Shot Prompting: You can ask Gemini to identify complex, arbitrary concepts (e.g., "Highlight the vintage cars manufactured before 1970 in this crowded lot") without needing to train a custom dataset.

Contextual Understanding: It understands intent, humor, visual metaphors, and multi-step instructions within the image.

Where Specialized Models (SAM, Mask R-CNN) Still Dominate
While general foundation models can output bounding boxes or rough pixel coordinates, specialized segmentation models remain vastly superior in three key areas:

Pixel-Perfect Accuracy: Models like SAM 3 are engineered specifically for dense geometric prediction. They deliver crisp, sub-pixel mask boundaries around delicate objects (like individual leaves or medical scans) where generalist LLMs produce rougher approximations.

Speed & Latency: An autonomous driving vehicle or a surgical robot cannot wait 1–2 seconds for a cloud-based LLM to reason through a prompt. Specialized vision models run locally on edge hardware in 5 to 30 milliseconds.

Compute Efficiency & Cost: Running a massive 100B+ parameter generalist model just to outline trees in millions of satellite frames is economically impractical compared to running a tiny, dedicated vision model.

Gemini acts as the "Brain" (Reasoning Layer): It interprets the user's high-level intent (e.g., "Find all trees that are leaning dangerously close to power lines").

Specialized Models act as the "Hands" (Execution Layer): Gemini passes coordinates or target descriptions to a model like SAM or a local segmentation head to generate the exact, pixel-perfect mask at high speed.




## Convolutional Neural Networks

CNNs were the dominant architecture for computer vision for a long time. They are built on the principle of locality.

Imagine you have a large grid of data (an image). A CNN passes a small "window" (called a kernel or filter) over the data, sliding it pixel by pixel. It performs a mathematical calculation (dot product) at each stop to detect simple features like edges or curves. As you go deeper into the network, these simple features combine into complex shapes (textures, objects).

The network assumes that a pixel is most related to its immediate neighbors.

Transformers are the architecture that made LLMs possible. They introduced a mechanism called Self-Attention. unlike CNNs, Transformers do not assume locality. They assume anything can be related to anything else, regardless of distance. Instead of sliding a window, the Transformer looks at the entire sequence of data at once. It assigns a "weight" (importance) to every part of the input relative to every other part.

## Object Detection with Keras and RetinaNet

Key Learning Objectives:

1. Understand why general-purpose pre-trained models (e.g., COCO) need domain adaptation.
2. Load and parse real scientific annotations (COCO format) downloaded via KaggleHub.
3. Perform visual diagnostics: Plotting ground-truth bounding boxes and predictions.
4. Adapt a RetinaNet object detector to a custom scientific domain taxonomy.
5. Fine-tune the model with Keras 3 and KerasHub using transfer learning.


In [ ]:
import os
# Configure Keras to use JAX backend CPU/GPU/TPU execution
os.environ["KERAS_BACKEND"] = "jax"

# For image manipulation and plotting
import json
import numpy as np
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import keras
import keras_hub
import kagglehub


In [ ]:
IMAGE_SIZE = 640 # Native 640
BBOX_FORMAT = "xywh"  # [xmin, ymin, width, height] in pixel coordinates
DOMAIN_CLASSES = ["Tumor"]  # Custom domain target category
NUM_CLASSES = len(DOMAIN_CLASSES)

# Select a small representative subset for fast demonstration
SUBSET_SIZE = 4

In [ ]:
# Data Wrangling Step

# Public microscopy dataset (Brain tumor / cellular structure detection)
dataset_path = kagglehub.dataset_download("pkdarabi/brain-tumor-image-dataset-semantic-segmentation")
print(f" {dataset_path}")

In [ ]:
# Get the text file
anno_path = os.path.join(dataset_path, "train", "_annotations.coco.json")
with open(anno_path, "r") as f:
    coco_data = json.load(f)

# Map image metadata and organize annotations by image ID
img_dict = {img["id"]: img for img in coco_data["images"]}
anno_dict = {}
for anno in coco_data["annotations"]:
    img_id = anno["image_id"]
    bbox = anno["bbox"]  # [xmin, ymin, width, height]
    category_id = anno["category_id"]
    if img_id not in anno_dict:
        anno_dict[img_id] = []
    anno_dict[img_id].append((bbox, category_id))

print(f"Total annotated images found in COCO file: {len(anno_dict)}")

# Subset the data (just to make the example go faster)
selected_img_ids = list(anno_dict.keys())[:SUBSET_SIZE]

images_list = []
boxes_list = []
labels_list = []

# Determine maximum bounding boxes per image in subset for uniform padding
max_boxes = max(len(anno_dict[img_id]) for img_id in selected_img_ids)

for img_id in selected_img_ids:
    img_info = img_dict[img_id]
    file_name = img_info["file_name"]
    img_file_path = os.path.join(dataset_path, "train", file_name)
    
    # Load and resize image to standard input dimensions (640x640)
    pil_img = Image.open(img_file_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    orig_w, orig_h = img_info["width"], img_info["height"]
    scale_x = IMAGE_SIZE / orig_w
    scale_y = IMAGE_SIZE / orig_h
    
    img_boxes = []
    img_labels = []
    for bbox, _ in anno_dict[img_id]:
        xmin, ymin, w, h = bbox
        # Rescale bounding box coordinates to match the 640x640 resized image
        img_boxes.append([xmin * scale_x, ymin * scale_y, w * scale_x, h * scale_y])
        img_labels.append(0)  # Index 0 corresponding to DOMAIN_CLASSES[0]
        
    # Pad fixed-size array (-1 for background/padding label)
    while len(img_boxes) < max_boxes:
        img_boxes.append([0.0, 0.0, 0.0, 0.0])
        img_labels.append(-1)
        
    images_list.append(np.array(pil_img, dtype="float32"))
    boxes_list.append(img_boxes)
    labels_list.append(img_labels)

train_images = np.array(images_list, dtype="float32")
train_boxes = np.array(boxes_list, dtype="float32")
train_classes = np.array(labels_list, dtype="int32")

train_targets = {
    "boxes": train_boxes,
    "labels": train_classes,
}

print(f"Images: {train_images.shape}")
print(f"Bounding Boxes: {train_boxes.shape}")
print(f"Class Labels: {train_classes.shape}\n")

In [ ]:
# -----------------------------------------------------------------------------
# Diagnostic Visualisation Helper Function
# -----------------------------------------------------------------------------
def plot_bounding_boxes(image, boxes, labels=None, class_names=None, title="Diagnostic Plot"):
    """
    Renders an image with overlay bounding boxes for visual diagnostic checks.
    """
    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    img_disp = np.array(image, dtype="uint8")
    h_img, w_img = img_disp.shape[:2]
    ax.imshow(img_disp)
    
    colors = ["#4285F4", "#EA4335", "#FBBC04", "#34A853"]
    if boxes is not None and len(boxes) > 0:
        for idx, box in enumerate(boxes):
            # Ensure coordinates are finite numbers (filter out NaN/Inf from early predictions)
            if not np.isfinite(box[:4]).all():
                continue
            xmin, ymin, w, h = float(box[0]), float(box[1]), float(box[2]), float(box[3])
            
            # Bound boxes within realistic image dimensions to prevent Matplotlib transform overflow
            if w <= 0 or h <= 0 or xmin < -w_img or ymin < -h_img or xmin > w_img * 2 or ymin > h_img * 2 or w > w_img * 2 or h > h_img * 2:
                continue
            
            # Clip rectangle coordinates for clean visualization display
            xmin_c = max(0.0, min(float(w_img - 1), xmin))
            ymin_c = max(0.0, min(float(h_img - 1), ymin))
            w_c = max(1.0, min(float(w_img - xmin_c), w))
            h_c = max(1.0, min(float(h_img - ymin_c), h))
            
            color = colors[idx % len(colors)]
            rect = patches.Rectangle(
                (xmin_c, ymin_c), w_c, h_c,
                linewidth=2.5, edgecolor=color, facecolor="none"
            )
            ax.add_patch(rect)
            
            if labels is not None and idx < len(labels):
                lbl_idx = int(labels[idx])
                if lbl_idx >= 0:
                    lbl_text = class_names[lbl_idx] if class_names and lbl_idx < len(class_names) else f"Class {lbl_idx}"
                    ax.text(
                        xmin_c, max(15.0, ymin_c - 5.0), lbl_text,
                        color="white", fontsize=10, weight="bold",
                        bbox=dict(boxstyle="square,pad=0.2", facecolor=color, alpha=0.85)
                    )
    ax.set_title(title, fontsize=13, weight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_bounding_boxes(
    image=train_images[0],
    boxes=train_boxes[0],
    labels=train_classes[0],
    class_names=DOMAIN_CLASSES,
    title="Ground Truth Example",
    output_path="04_cv3_ground_truth.png",
)

In [ ]:
# -----------------------------------------------------------------------------
# Zero-Shot Baseline Inference 
# -----------------------------------------------------------------------------
baseline_detector = keras_hub.models.ObjectDetector.from_preset(
    "retinanet_resnet50_fpn_v2_coco",
    bounding_box_format=BBOX_FORMAT,
)

In [ ]:
# example zero-shot image (for known classes)
image_url = "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"

# Load the image and get it in the correct format
image_path = keras.utils.get_file(origin=image_url)
image = keras.utils.load_img(image_path)
image_batch = np.expand_dims(keras.utils.img_to_array(image), axis=0)

# Make the predcition!
coco_preds = baseline_detector.predict(image_batch)

plot_bounding_boxes(
    image=coco_img_np[0],
    boxes=coco_preds["boxes"][0],
    labels=coco_preds["labels"][0],
    title="RetinaNet image it has a class for",
)

In [ ]:


# 4B. Demo on specialized scientific domain image (brain tumor / microscopy)
print("  - Testing baseline detector on unfamiliar scientific domain image...")
baseline_preds = baseline_detector.predict(train_images[:1])
print("  - Pre-trained model output keys:", list(baseline_preds.keys()))
print(f"  - Number of raw detections: {baseline_preds['boxes'].shape[1]}")

# Filter detections by confidence threshold (> 0.25)
conf_scores_base = np.array(baseline_preds["confidence"][0])
conf_mask = (conf_scores_base > 0.25) & np.isfinite(conf_scores_base)
filtered_boxes = baseline_preds["boxes"][0][conf_mask]
filtered_labels = baseline_preds["labels"][0][conf_mask]
print(f"    * Detections on Scientific Domain Image (conf > 0.25): {len(filtered_boxes)}")
print("    * Notice: Pre-trained COCO model yields 0 detections on specialized scientific data!")

plot_bounding_boxes(
    image=train_images[0],
    boxes=filtered_boxes,
    labels=filtered_labels,
    title="Diagnostic 2b: Base Model on Unseen Domain Data (No Detections)",
    output_path="04_cv3_pre_tuning_domain_unknown.png",
)